# 03 - Interpretacion de Perfiles, Anatomia de Centroides y Explicabilidad en K-Means

## Curso de Machine Learning: Modelos No Supervisados

Obtener la etiqueta numerica de un cluster (`0`, `1`, `2`) carece de valor practico por si misma. En un proyecto real de Ciencia de Datos, el verdadero entregable del agrupamiento no es el vector de asignaciones `model.labels_`, sino la **caracterizacion semantica y cuantitativa de cada grupo** (¿Quienes conforman el Cluster 0? ¿Que rasgos fisicos, quimicos o de comportamiento distinguen al Cluster 1 del Cluster 2?).

En este cuaderno desarrollaremos una metodologia sistematica de **explicabilidad para modelos de clustering**, combinando el analisis geometrico de los centroides en el espacio estandarizado ($Z$-scores) con la reconstruccion en unidades originales, pruebas estadisticas de relevancia de variables y modelos sustitutos supervisados.

---

### Contenidos
1. Anatomia de los Centroides: Espacio Estandarizado ($Z$-Scores) vs. Escala Original (`inverse_transform`)
2. Mapas de Calor de Perfiles de Centroides (Cluster Heatmaps)
3. Graficos Radiales (Spider / Radar Charts) para Comunicacion Ejecutiva
4. Diagramas de Coordenadas Paralelas Multivariadas
5. Importancia de Variables en Clustering: Estadistico $F$ de ANOVA Univariado
6. Explicabilidad mediante Modelos Sustitutos: Arbol de Decision Clasificador (Cluster Surrogate Tree)
7. Proyeccion y Visualizacion en 2D mediante Analisis de Componentes Principales (PCA)
8. Sintesis y Bautizo de Arquetipos (Cluster Personas)
9. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)

# Preparacion del pipeline optimo (K=3) sobre el Dataset Wine
wine = load_wine(as_frame=True)
X = wine.data.copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, init='k-means++', n_init=15, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

X_con_clusters = X.copy()
X_con_clusters['Cluster'] = clusters
print("Distribucion de muestras por Cluster descubierto:")
print(X_con_clusters['Cluster'].value_counts().sort_index())


## 1. Anatomia de los Centroides: Espacio $Z$ vs. Escala Original

Cuando entrenamos K-Means sobre datos transformados con `StandardScaler`, las coordenadas almacenadas en `kmeans.cluster_centers_` estan expresadas en **unidades de desviacion estandar respecto a la media poblacional ($Z$-scores)**:
- Un valor de $+1.20$ en el centroide indica que ese cluster tiene un promedio $1.20$ desviaciones estandar **por encima** de la media global en esa variable.
- Un valor de $-0.95$ indica que ese cluster esta $0.95$ desviaciones estandar **por debajo** de la media global.

Esto es extraordinariamente util para comparar variables heterogeneas en un mismo grafico. Sin embargo, para reportar hallazgos a expertos de dominio (enologos, medicos, ejecutivos financieros), debemos aplicar `scaler.inverse_transform(kmeans.cluster_centers_)` para recuperar las magnitudes fisicas reales (grados de alcohol, mg/L de magnesio, etc.).


In [ ]:
# 1. Centroides en espacio estandarizado (Z-scores)
df_centroides_z = pd.DataFrame(
    kmeans.cluster_centers_, 
    columns=X.columns,
    index=[f'Cluster {i}' for i in range(3)]
)

# 2. Centroides reconstruidos en unidades fisicas originales
centroides_originales = scaler.inverse_transform(kmeans.cluster_centers_)
df_centroides_orig = pd.DataFrame(
    centroides_originales, 
    columns=X.columns,
    index=[f'Cluster {i}' for i in range(3)]
)

print("=" * 80)
print("CENTROIDES EN UNIDADES FISICAS ORIGINALES (Media de cada variable por Cluster)")
print("=" * 80)
print(df_centroides_orig.T.round(2))


## 2. Mapa de Calor de Perfiles de Centroides ($Z$-Scores)

Visualizar la matriz `df_centroides_z` mediante un mapa de calor divergente centrado en $0.0$ permite identificar de un solo vistazo la **firma caracteristica** de cada cluster: los bloques rojos intensos muestran los atributos donde el cluster sobresale positivamente, y los azules intensos donde presenta valores inusualmente bajos.


In [ ]:
plt.figure(figsize=(12, 6.5))
sns.heatmap(
    df_centroides_z.T, 
    annot=True, 
    fmt='.2f', 
    cmap='RdBu_r', 
    center=0, 
    linewidths=0.8,
    cbar_kws={'label': 'Desviaciones Estandar respecto a la Media Global (Z-Score)'}
)
plt.title('Firma Quimiometrica de cada Cluster (Centroides en Espacio Z)', fontsize=13, fontweight='bold')
plt.ylabel('Variables Fisico-Quimicas', fontsize=11, fontweight='bold')
plt.xlabel('Clusters Descubiertos', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Grafico Radial (Radar / Spider Chart) de Arquetipos

Los graficos radiales son una de las herramientas visuales mas efectivas para comparar simultaneamente la huella multivariada de los centroides sobre las variables mas discriminantes.


In [ ]:
# Seleccion de 6 variables clave para el Radar Chart
vars_radar = ['alcohol', 'malic_acid', 'flavanoids', 'color_intensity', 'hue', 'proline']
datos_radar = df_centroides_z[vars_radar].values

angulos = np.linspace(0, 2 * np.pi, len(vars_radar), endpoint=False).tolist()
datos_radar = np.concatenate((datos_radar, datos_radar[:, [0]]), axis=1)
angulos += angulos[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colores_cluster = ['#1f77b4', '#2ca02c', '#d62728']

for i in range(3):
    ax.plot(angulos, datos_radar[i], 'o-', linewidth=2.2, label=f'Cluster {i}', color=colores_cluster[i])
    ax.fill(angulos, datos_radar[i], alpha=0.18, color=colores_cluster[i])

ax.set_thetagrids(np.degrees(angulos[:-1]), vars_radar, fontsize=11, fontweight='bold')
ax.set_ylim(-1.5, 1.5)
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.title('Huella Multivariada de los Centroides (Radar Chart en Z-Scores)', fontsize=13, fontweight='bold', y=1.08)
plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1), fontsize=11)
plt.tight_layout()
plt.show()


## 4. Importancia de Variables en Clustering: Estadistico $F$ de ANOVA

A diferencia de un Arbol de Decision, K-Means no posee un atributo nativo `.feature_importances_`. ¿Como sabemos cuales de las 13 variables fueron las verdaderas responsables de separar los grupos y cuales son puro ruido?

Podemos cuantificar el poder discriminante de cada variable $x_j$ calculando el **Estadistico $F$ de Analisis de Varianza (ANOVA) de una via** entre los clusters:

$$F_j = \frac{\text{Varianza Inter-Cluster de } x_j}{\text{Varianza Intra-Cluster de } x_j} = \frac{\sum_{k=1}^{K} n_k (\bar{x}_{kj} - \bar{x}_j)^2 / (K - 1)}{\sum_{k=1}^{K} \sum_{i \in C_k} (x_{ij} - \bar{x}_{kj})^2 / (n - K)}$$

Un valor $F_j$ elevado indica que la variable $x_j$ presenta medias radicalmente distintas entre los clusters y muy poca dispersion dentro de cada cluster.


In [ ]:
# Calculo del Estadistico F de ANOVA para cada variable
f_values, p_values = f_classif(X_scaled, clusters)

df_importancia_anova = pd.DataFrame({
    'Variable': X.columns,
    'F_Statistic': f_values,
    'p_value': p_values
}).sort_values('F_Statistic', ascending=True)

plt.figure(figsize=(10, 6))
colores_bar = plt.cm.viridis(np.linspace(0.2, 0.9, len(df_importancia_anova)))
plt.barh(df_importancia_anova['Variable'], df_importancia_anova['F_Statistic'], color=colores_bar, edgecolor='k')
plt.xlabel('Estadistico F de ANOVA (Razon de Varianza Inter/Intra Cluster)', fontsize=11, fontweight='bold')
plt.title('Poder Discriminante de cada Variable en la Particion K-Means', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 5 Variables que mas diferencian a los Clusters:")
print(df_importancia_anova.sort_values('F_Statistic', ascending=False).head(5).round(4).to_string(index=False))


## 5. Explicabilidad mediante Modelos Sustitutos (Surrogate Decision Tree)

Una tecnica sumamente poderosa en la industria para traducir fronteras geometricas de K-Means en **reglas de negocio legibles por humanos** consiste en entrenar un **Arbol de Decision de Clasificacion poco profundo (`max_depth=2` o `3`)** sobre las variables en su escala original $X$, utilizando como variable objetivo las etiquetas generadas por K-Means (`clusters`).

De esta forma combinamos el descubrimiento geometrico multivariado de K-Means con la explicabilidad de reglas `IF-THEN` de CART.


In [ ]:
# Entrenamiento de un Arbol Clasificador Sustituto sobre las etiquetas del Cluster
surrogate_tree = DecisionTreeClassifier(max_depth=2, random_state=42)
surrogate_tree.fit(X, clusters)

fidelidad = surrogate_tree.score(X, clusters)
print(f"Fidelidad del Arbol Sustituto (max_depth=2) replicando a K-Means: {fidelidad*100:.2f}%\n")
print("Reglas de Negocio Extraidas para Explicar los Clusters:")
print("=" * 65)
print(export_text(surrogate_tree, feature_names=list(X.columns), decimals=2))

plt.figure(figsize=(14, 6))
plot_tree(surrogate_tree, feature_names=list(X.columns), 
          class_names=[f'Cluster {i}' for i in range(3)],
          filled=True, rounded=True, fontsize=10, precision=2)
plt.title(f'Arbol de Decision Sustituto para Explicar los Clusters (Fidelidad = {fidelidad*100:.1f}%)', 
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Proyeccion Bidimensional mediante PCA

Finalmente, dado que nuestros centroides viven en un espacio de $\mathbb{R}^{13}$, podemos proyectar tanto las observaciones como los centroides sobre el plano formado por las **dos primeras Componentes Principales (PCA)** que capturan la mayor varianza direccional del dataset.


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
centroides_pca = pca.transform(kmeans.cluster_centers_)

var_explicada = pca.explained_variance_ratio_ * 100

plt.figure(figsize=(10, 6.5))
for i, color in enumerate(['#1f77b4', '#2ca02c', '#d62728']):
    mask = (clusters == i)
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, s=55, alpha=0.8, edgecolor='k', label=f'Cluster {i}')

plt.scatter(centroides_pca[:, 0], centroides_pca[:, 1], c='yellow', marker='*', s=400, 
            edgecolor='black', linewidth=2, label='Centroides Proyectados')

plt.xlabel(f'Componente Principal 1 ({var_explicada[0]:.1f}% de varianza)', fontsize=11, fontweight='bold')
plt.ylabel(f'Componente Principal 2 ({var_explicada[1]:.1f}% de varianza)', fontsize=11, fontweight='bold')
plt.title('Visualizacion Topologica de los 3 Clusters en el Espacio de Componentes Principales (PCA)', 
          fontsize=12, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 7. Sintesis y Bautizo de Arquetipos (Cluster Personas)

Integrando el mapa de calor $Z$, la tabla en unidades originales y el arbol sustituto, podemos bautizar formalmente cada grupo descubierto:

1. **Perfil reserva estructurado y de alta guarda**: Vinos con el mayor grado alcoholico ($\approx 13.7\%$), altisima concentracion de flavonoides ($\approx 2.98$), alta prolina ($\approx 1100$ mg/L) y baja acidez malica.
2. **Perfil joven, ligero y fresco**: Vinos de baja graduacion alcoholica ($\approx 12.2\%$), baja intensidad de color ($\approx 3.0$), baja prolina y tonalidad suave.
3. **Perfil intenso, acido y astringente**: Vinos con altisima acidez malica ($\approx 3.33$ g/L), muy baja concentracion de flavonoides ($\approx 0.82$, responsables del amargor no flavonoide) y la mayor intensidad cromatica ($\approx 7.39$).

---

## 8. Conclusiones

### Puntos Clave:
1. **Doble Lectura de Centroides**: Analiza siempre los centroides en espacio $Z$ para comparar importancias relativas entre variables, y en escala original (`inverse_transform`) para interpretar magnitudes reales de negocio.
2. **Ranking de Variables por ANOVA**: El estadistico $F$ univariado permite ordenar rapidamente que variables impulsaron la formacion de los grupos.
3. **Arboles Sustitutos (Surrogate Models)**: Entrenar un `DecisionTreeClassifier` superficial sobre las etiquetas de K-Means permite extraer reglas simples con mas del 90% de fidelidad para implementar en sistemas productivos o explicar a comites de negocio.

En el siguiente cuaderno, **04_Ejercicios_Practicos.ipynb**, aplicaras todo este flujo metodologico sobre un problema de alto impacto socioeconomico y financiero.
